In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"



## Load the model as well as the tokenizer

In [9]:
from peft import set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from unsloth import FastLanguageModel
def load_adapter(huggingface_repo):
    model,tokenizer=FastLanguageModel.from_pretrained(
        model_name="unsloth/gemma-3-270m-it",
        max_seq_length=2048,
        load_in_4bit=True,
    )
    model=FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "up_proj","down_proj","gate_proj"],
        lora_alpha=32,
        lora_dropout=0,
        use_rslora=False,
    )
    try:
        model_weights=hf_hub_download(
         repo_id=f"Srishtik/{huggingface_repo}",
         filename="adapter_model.safetensors"
        )
    except:
        model_weights=hf_hub_download(
         repo_id=f"Srishtik/{huggingface_repo}",
         filename="adapter_model.bin"
        )
    from safetensors.torch import load_file
    model_weights=load_file(model_weights)
    set_peft_model_state_dict(model,model_weights)
    return model,tokenizer

In [10]:
full_model,full_tokenizer=load_adapter("gemma-ag-news-finetuned_on_50K_samples")

==((====))==  Unsloth 2026.6.1: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

## Evaluation Pipeline

#### 1)Evaluate the models on test dataset on batch size of 8. 

#### 2)This loads all the models that were merged with merging techniques like linear,svd,ties,dare and slerp.

In [8]:
import torch
import gc
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# ── AG News label map ──
LABEL_MAP = {
    "1": "World",
    "2": "Sports", 
    "3": "Business",
    "4": "Sci/Tech",
    "World": "World",
    "Sports": "Sports",
    "Business": "Business",
    "Sci/Tech": "Sci/Tech",
}

INT_TO_LABEL = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

def format_prompt(title: str, description: str) -> str:
    return (
        f"Classify the following news article into one of these categories: "
        f"World, Sports, Business, Sci/Tech.\n\n"
        f"Title: {title}\n"
        f"Description: {description}\n\n"
        f"Category:"
    )

def extract_label(generated_text: str) -> str:
    """Extract the predicted label from generated text."""
    text = generated_text.strip()
    for label in ["Sci/Tech", "Business", "Sports", "World"]:  # longer first to avoid partial match
        if label.lower() in text.lower():
            return label
    return "World"  # fallback


def evaluate_on_agnews(
    repo_id: str,
    tokenizer,
    num_samples: int = 500,
    batch_size: int = 8,
    max_new_tokens: int = 10,
    device: str = "cuda",
) -> dict:
    """
    Evaluate a merged model on AG News test set.

    Args:
        repo_id       : HuggingFace repo to evaluate
        tokenizer     : shared tokenizer
        num_samples   : number of test samples (full test = 7600)
        batch_size    : inference batch size
        max_new_tokens: how many tokens to generate for label
        device        : cuda or cpu

    Returns:
        dict with accuracy, macro_f1, per_class_f1, repo_id
    """
    print(f"\n{'─'*60}")
    print(f"Evaluating: {repo_id}")
    print(f"{'─'*60}")

    # ── Load model ──
    model = AutoModelForCausalLM.from_pretrained(
        repo_id,
        torch_dtype=torch.float16,
        device_map=device,
    )
    model.eval()

    # ── Load dataset ──
    dataset = load_dataset("ag_news", split="test")
    dataset = dataset.select(range(num_samples))

    preds  = []
    labels = []

    # ── Inference in batches ──
    for i in tqdm(range(0, len(dataset), batch_size), desc=repo_id.split("/")[-1]):
        batch = dataset[i : i + batch_size]

        prompts = [
            format_prompt(title, desc)
            for title, desc in zip(batch["text"], batch["text"])  # ag_news has no separate title field
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,        # greedy for reproducibility
                pad_token_id=tokenizer.eos_token_id,
            )

        # Decode only the generated part (strip the prompt)
        for j, output in enumerate(outputs):
            input_len  = inputs["input_ids"].shape[1]
            generated  = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_label = extract_label(generated)
            true_label = INT_TO_LABEL[batch["label"][j]]

            preds.append(pred_label)
            labels.append(true_label)

    # ── Compute metrics ──
    label_names   = ["World", "Sports", "Business", "Sci/Tech"]
    accuracy      = accuracy_score(labels, preds)
    macro_f1      = f1_score(labels, preds, average="macro",    labels=label_names, zero_division=0)
    per_class_f1  = f1_score(labels, preds, average=None,       labels=label_names, zero_division=0)

    result = {
        "repo_id"      : repo_id,
        "accuracy"     : round(accuracy, 4),
        "macro_f1"     : round(macro_f1, 4),
        "per_class_f1" : {
            label: round(float(score), 4)
            for label, score in zip(label_names, per_class_f1)
        },
        "num_samples"  : num_samples,
    }

    print(f"  Accuracy  : {result['accuracy']:.4f}")
    print(f"  Macro F1  : {result['macro_f1']:.4f}")
    for label, score in result["per_class_f1"].items():
        print(f"  F1 {label:<10}: {score:.4f}")

    # ── Free memory ──
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return result


def evaluate_all_models(
    repos: list[str],
    tokenizer,
    num_samples: int = 500,
    batch_size: int = 8,
) -> dict:
    """
    Evaluate all merged models sequentially, freeing memory between each.

    Args:
        repos       : list of HuggingFace repo ids
        tokenizer   : shared tokenizer
        num_samples : test samples per model
        batch_size  : inference batch size

    Returns:
        dict mapping repo_id → metrics
    """
    all_results = {}

    for repo in repos:
        result = evaluate_on_agnews(
            repo_id     = repo,
            tokenizer   = tokenizer,
            num_samples = num_samples,
            batch_size  = batch_size,
        )
        all_results[repo] = result

    # ── Summary table ──
    print(f"\n{'═'*60}")
    print(f"{'MODEL':<35} {'ACC':>6} {'F1':>6}")
    print(f"{'─'*60}")
    for repo, r in all_results.items():
        name = repo.split("/")[-1]
        print(f"{name:<35} {r['accuracy']:>6.4f} {r['macro_f1']:>6.4f}")
    print(f"{'═'*60}")

    return all_results


# ── Usage ──

tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-3-270m-it")

repos = [
    "Srishtik/gemma-3-270m-linear-merged",
    "Srishtik/gemma-3-270m-svd-merged",
    "Srishtik/gemma-3-270m-ties-merged",
    "Srishtik/gemma-3-270m-dare-merged",
    "Srishtik/gemma-3-270m-slerp-merged",
]

all_results = evaluate_all_models(
    repos       = repos,
    tokenizer   = tokenizer,
    num_samples = 500,    # increase to 7600 for full test set
    batch_size  = 8,
)



────────────────────────────────────────────────────────────
Evaluating: Srishtik/gemma-3-270m-linear-merged
────────────────────────────────────────────────────────────


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

gemma-3-270m-linear-merged: 100%|██████████| 63/63 [00:28<00:00,  2.23it/s]


  Accuracy  : 0.2600
  Macro F1  : 0.2652
  F1 World     : 0.3102
  F1 Sports    : 0.1754
  F1 Business  : 0.3756
  F1 Sci/Tech  : 0.1995

────────────────────────────────────────────────────────────
Evaluating: Srishtik/gemma-3-270m-svd-merged
────────────────────────────────────────────────────────────


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

gemma-3-270m-slerp-merged: 100%|██████████| 63/63 [00:28<00:00,  2.21it/s]


  Accuracy  : 0.3440
  Macro F1  : 0.2897
  F1 World     : 0.4346
  F1 Sports    : 0.5278
  F1 Business  : 0.0526
  F1 Sci/Tech  : 0.1436

════════════════════════════════════════════════════════════
MODEL                                  ACC     F1
────────────────────────────────────────────────────────────
gemma-3-270m-linear-merged          0.2600 0.2652
gemma-3-270m-svd-merged             0.2720 0.2816
gemma-3-270m-ties-merged            0.3720 0.3741
gemma-3-270m-dare-merged            0.2660 0.2444
gemma-3-270m-slerp-merged           0.3440 0.2897
════════════════════════════════════════════════════════════


In [14]:
def format_prompt(text: str) -> str:
    return (
        f"Classify the following news article into one of these categories: "
        f"World, Sports, Business, Sci/Tech.\n\n"
        f"Article: {text}\n\n"
        f"Category:"
    )

## Evaluating the fully finetuned model 

#### 1) Since this model is already loaded I created a separate evaluation for this

In [15]:
def evaluate_initialized_model(
    model,
    tokenizer,
    model_name: str = "full_model",
    num_samples: int = 500,
    batch_size: int = 8,
    max_new_tokens: int = 10,
    device: str = "cuda",
) -> dict:
    """
    Evaluate an already-loaded model on AG News.
    Does NOT load or delete the model — caller manages memory.
    """
    print(f"\n{'─'*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'─'*60}")

    model.eval()

    dataset = load_dataset("ag_news", split="test")
    dataset = dataset.select(range(num_samples))

    preds  = []
    labels = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=model_name):
        batch = dataset[i : i + batch_size]

        prompts = [format_prompt(text) for text in batch["text"]]

        inputs = tokenizer(
            prompts,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )

        for j, output in enumerate(outputs):
            input_len  = inputs["input_ids"].shape[1]
            generated  = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_label = extract_label(generated)
            true_label = INT_TO_LABEL[batch["label"][j]]

            preds.append(pred_label)
            labels.append(true_label)

    label_names  = ["World", "Sports", "Business", "Sci/Tech"]
    accuracy     = accuracy_score(labels, preds)
    macro_f1     = f1_score(labels, preds, average="macro", labels=label_names, zero_division=0)
    per_class_f1 = f1_score(labels, preds, average=None,    labels=label_names, zero_division=0)

    result = {
        "repo_id"      : model_name,
        "accuracy"     : round(accuracy, 4),
        "macro_f1"     : round(macro_f1, 4),
        "per_class_f1" : {
            label: round(float(score), 4)
            for label, score in zip(label_names, per_class_f1)
        },
        "num_samples"  : num_samples,
    }

    print(f"  Accuracy  : {result['accuracy']:.4f}")
    print(f"  Macro F1  : {result['macro_f1']:.4f}")
    for label, score in result["per_class_f1"].items():
        print(f"  F1 {label:<10}: {score:.4f}")

    return result

In [16]:
result = evaluate_initialized_model(
    model      = full_model,
    tokenizer  = full_tokenizer,
    num_samples = 500,
    batch_size  = 8,
)


────────────────────────────────────────────────────────────
Evaluating: full_model
────────────────────────────────────────────────────────────


full_model: 100%|██████████| 63/63 [04:59<00:00,  4.75s/it] 

  Accuracy  : 0.3780
  Macro F1  : 0.2872
  F1 World     : 0.4277
  F1 Sports    : 0.6404
  F1 Business  : 0.0513
  F1 Sci/Tech  : 0.0294
